# Public Mail — run the entire collection in Google Colab

**No local Python, Docker, VPS, public dashboard, or GPU is required.** This notebook installs the real scraper in an isolated environment, imports your private CSV/SQLite input, crawls public websites, saves resumable checkpoints to Google Drive, and exports emails **with public source URLs**.

### First run
1. Select a **CPU Python runtime**. Keep **Use Google Drive** enabled below.
2. Choose **Runtime → Run all**, approve the Drive mount, and upload your original `domains.csv` when asked. The default is a **100-claim pilot**, not the whole list.
3. Inspect the results. In **Start / resume**, select **full**, then run that cell again to process the saved queue. No second upload or local process is needed.

### After a Colab runtime reset
Open this notebook again, keep the **same project name**, and run all. It restores the newest valid Drive snapshot and reuses the saved queue/input. Then select **full** again to continue. Do not start a new project just to resume.

**Important:** 33,144 input domains are not 33,144 guaranteed email addresses. Some sites are offline, blocked, unrelated, or contact-form-only. Counts and gaps are reported honestly. This workflow never guesses addresses or bypasses robots, CAPTCHA, or access controls.

Colab can execute remotely without your PC doing the scraping, but its runtime can expire or be stopped, including after browser disconnection. **It is not guaranteed 24/7 hosting.** An abrupt reset can lose work since the last *completed, verified* Drive snapshot. No keep-alive/limit-evasion scripts are used. Use only one runtime per Drive project.

Your original CSV is **not in the public GitHub repository**; upload it once or select its private Drive path. Clear notebook outputs before sharing, because outputs/results may contain personal data.

In [ ]:
# @title 1. Collection settings
PROJECT_NAME = "publicmail-33144"  # @param {type:"string"}
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
INPUT_MODE = "Upload or reuse saved queue"  # @param ["Upload or reuse saved queue", "Upload another input file", "Use a Drive file path"]
DRIVE_INPUT_PATH = ""  # @param {type:"string"}
SQLITE_INPUT_TABLE = ""  # @param {type:"string"}
ENABLE_BROWSER = True  # @param {type:"boolean"}
CONCURRENCY = 16  # @param {type:"integer"}
REQUEST_DELAY_SECONDS = 1.5  # @param {type:"number"}
MAX_PAGES_PER_SITE = 24  # @param {type:"integer"}
MAX_DEPTH = 3  # @param {type:"integer"}
EXPECTED_DOMAINS = 33144  # @param {type:"integer"}
CODE_REF = "main"  # @param {type:"string"}

from pathlib import Path
CONTENT_ROOT = Path("/content")
print("CPU runtime is sufficient. Browser fallback uses Chromium, not a GPU.")
print("Keep the project name unchanged when resuming the same collection.")

## 2. Install the application — only inside Colab

This can take several minutes, especially Chromium's first installation. Application dependencies go into a **private virtual environment**, not Colab's notebook kernel. Setup can be rerun without deleting data. Source code lives separately from your private input/state.

The default code ref is `main`. Before the Colab pull request is merged, setup transparently uses the published Colab feature branch if `main` does not contain the notebook runner yet. It prints the exact revision used. An existing runtime checkout is reused rather than reset; start a fresh runtime to update code, with Drive backups enabled.

In [ ]:
# @title Install / verify the private runtime
import os
import re
import subprocess
import sys

if "client" in globals():
    client._idle()  # Do not replace packages underneath an active worker.
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", PROJECT_NAME):
    raise ValueError("Project name must use letters/numbers/underscore/hyphen, up to 64 characters.")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_./-]{0,127}", CODE_REF):
    raise ValueError("Choose a valid Git branch or tag, for example main.")
CONTENT_ROOT.mkdir(parents=True, exist_ok=True)
REPOSITORY = CONTENT_ROOT / "publicmail-code"
REPO_URL = "https://github.com/FredTheFork/emailscraperfromsites.git"
COLAB_FEATURE_REF = "arena/ff9cb618-emailscraperfromsites"
GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if not REPOSITORY.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", CODE_REF, REPO_URL, str(REPOSITORY)], check=True, env=GIT_ENV)
elif not (REPOSITORY / ".git").is_dir():
    raise RuntimeError("The code folder already exists but is not a Git checkout. Do not overwrite it; use a fresh runtime.")
if not (REPOSITORY / "notebooks" / "colab_client.py").is_file():
    if CODE_REF != "main":
        raise RuntimeError("This code ref does not include the Colab workflow. Select main after the Colab PR is merged.")
    print("main does not include the Colab workflow yet; using the published feature branch.")
    subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "--depth", "1", "origin", COLAB_FEATURE_REF], check=True, env=GIT_ENV)
    subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach", "FETCH_HEAD"], check=True, env=GIT_ENV)
CODE_REVISION = subprocess.check_output(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], text=True).strip()
print("Using code revision:", CODE_REVISION)
sys.path.insert(0, str(REPOSITORY / "notebooks"))
from colab_client import ColabClient, install_runtime
PYTHON, APP_ENV = install_runtime(REPOSITORY, CONTENT_ROOT, browser=ENABLE_BROWSER)
LOCAL_PROJECT = CONTENT_ROOT / "publicmail-workspaces" / PROJECT_NAME
LOCAL_PROJECT.mkdir(parents=True, exist_ok=True)
print("Private runtime ready:", PYTHON)

## 3. Mount Drive and restore saved progress

Approve the normal Google Drive authorization dialog in Colab. Your project is stored privately under **MyDrive / PublicMail / project-name**. Share neither that folder nor its results publicly.

The **active SQLite database stays on `/content`**, Colab's local disk. Drive receives verified, closed snapshots—not an active WAL database. Three good snapshot generations are kept. A session marker catches many accidental duplicate runtimes, but Drive is not a distributed lock service: **never run two Colab runtimes on this project simultaneously**.

If Drive is disabled, all files are ephemeral and cannot survive a runtime reset. This notebook does not create a fake backup folder when the required Drive mount is missing.

In [ ]:
# @title Mount Drive / restore the queue
import json

DRIVE_MOUNT = None
DRIVE_PROJECT = None
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    DRIVE_MOUNT = CONTENT_ROOT / "drive"
    drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not (DRIVE_MOUNT / "MyDrive").is_dir():
        raise RuntimeError("Drive is not mounted correctly. Re-run this cell; do not proceed without your backups.")
    DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "PublicMail" / PROJECT_NAME
else:
    print("WARNING: Drive is disabled. A runtime reset loses the input, database, snapshots, and exports.")

client = ColabClient(PYTHON, LOCAL_PROJECT, drive_dir=DRIVE_PROJECT, drive_mount=DRIVE_MOUNT, environment=APP_ENV)
health = client.json("doctor", *(["--check-browser"] if ENABLE_BROWSER else []))
print("Environment:", json.dumps(health, indent=2))
PREPARED = client.json("prepare")
print("Restored snapshot:", PREPARED["restored"] or "Existing local state / new collection")
print("Backup storage:", PREPARED["backup_dir"])
print("Domains saved:", format(PREPARED["stats"]["total"], ","))
print("Queue is paused. No target-site crawl has started from this cell.")

## 4. Import your actual CSV or SQLite list

On the first run, select your original **`domains.csv` containing 33,144 domains**. It is uploaded to the Colab runtime, imported in repeat-safe batches, and copied privately to your Drive project. A resumed queue already contains its original metadata and does not require another upload.

For a large SQLite file, place a **consistent backup/export** in Drive and select **Use a Drive file path** above. Do not upload only the main file of a database whose writer still has an uncheckpointed WAL. Input databases are copied onto local disk before querying and are never used as the working state DB.

In [ ]:
# @title Upload / reuse / import input
current = client.json("status")
input_path = None
if INPUT_MODE == "Use a Drive file path":
    if not DRIVE_INPUT_PATH.strip():
        raise ValueError("Enter the full private Drive input path in Collection settings, then rerun that settings cell.")
    input_path = Path(DRIVE_INPUT_PATH).expanduser()
elif INPUT_MODE == "Upload or reuse saved queue" and current["stats"]["total"]:
    print("Reusing", format(current["stats"]["total"], ","), "saved domains; no upload needed.")
elif INPUT_MODE == "Upload or reuse saved queue" and PREPARED.get("saved_input"):
    input_path = Path(PREPARED["saved_input"])
else:
    from google.colab import files
    uploads_dir = LOCAL_PROJECT / "uploads"
    uploads_dir.mkdir(exist_ok=True)
    uploaded = files.upload(target_dir=str(uploads_dir))
    if len(uploaded) != 1:
        raise ValueError("Choose exactly one CSV or SQLite input, then rerun this cell.")
    input_path = uploads_dir / Path(next(iter(uploaded))).name
    del uploaded  # Do not hold a second copy of a large upload in notebook memory.
if input_path is not None:
    args = ["import", str(input_path)]
    if SQLITE_INPUT_TABLE.strip():
        args += ["--table", SQLITE_INPUT_TABLE.strip()]
    report = client.json(*args)
    print("Import report:", json.dumps(report, indent=2))
STATE = client.json("status")
total = STATE["stats"]["total"]
if not total:
    raise RuntimeError("No domains were imported. Read the import error; do not start an empty run.")
print("READY:", format(total, ","), "domains;", format(STATE["stats"]["unique_emails"], ","), "emails collected so far.")
if EXPECTED_DOMAINS and total != EXPECTED_DOMAINS:
    print("CHECK INPUT: expected", format(EXPECTED_DOMAINS, ","), "domains, but the saved queue contains", format(total, ","))
print("A verified recovery snapshot has been saved. Importing does not start crawling.")

## 5. Start / resume — this is the long-running cell

- **pilot** processes up to 100 site **claims**, including retries. Inspect evidence and blocked/no-email reasons before scaling up.
- **full** processes all pending sites and scheduled retries in this saved queue. Completed work is retained on reruns.
- **status only** makes no target-site requests.

Start with the pilot; to run the rest, select **full in this cell** and run this cell again. No need to repeat installation or upload. The default **five-hour session budget** is a planned, graceful stop, not a Colab runtime promise. Select full and run again to continue. Set hours to 0 for no planned stop; Colab can still end the runtime at any time.

Progress appears approximately every 30 seconds. Backups start before crawling, periodically during the run (default two minutes **after the previous backup completes**), and on normal/graceful exit. A failed/timed-out periodic backup stops new claims instead of silently running for hours without backups. After a run stops, aligned CSV/JSONL exports and a ZIP are generated locally and copied to Drive.

**To stop deliberately:** click this cell's stop button / Interrupt once and allow the runner to finish its final backup/export. Do not reset/delete the runtime to pause it. A forcibly killed runtime cannot execute cleanup.

Throughput and coverage depend on the sites and Colab's shared IP reputation. HTTP 429, robots, and access challenges are not bypassed. A scheduled Retry-After can legitimately leave a run waiting.

In [ ]:
# @title Start / resume the saved collection
RUN_MODE = "pilot"  # @param ["pilot", "full", "status only"]
SESSION_HOURS = 5.0  # @param {type:"number"}
BACKUP_EVERY_MINUTES = 2.0  # @param {type:"number"}

if RUN_MODE == "status only":
    print(json.dumps(client.json("status"), indent=2))
else:
    args = ["run", "--max-sites", "100" if RUN_MODE == "pilot" else "0",
            "--hours", str(SESSION_HOURS),
            "--checkpoint-seconds", str(BACKUP_EVERY_MINUTES * 60),
            "--concurrency", str(CONCURRENCY), "--delay", str(REQUEST_DELAY_SECONDS),
            "--max-pages", str(MAX_PAGES_PER_SITE), "--max-depth", str(MAX_DEPTH),
            "--browser" if ENABLE_BROWSER else "--no-browser"]
    client.run(*args)
    print("Run stopped safely. Review the status and download cells below.")

## 6. Inspect status and download results

The ZIP contains `emails.csv` (website-email associations), `unique_emails.csv`, `observations.csv`, exact `observations.jsonl`, `coverage.csv` (every website, including failures/no-email sites), and `summary.json`.

**Websites with an email**, **unique emails**, and **processed websites** are different measures. `completed` means at least one email was observed, not that every possible page was exhausted. Review provenance/relationship labels and limitations; an observed address is not deliverability, ownership, or marketing consent.

Results are available in Drive even if your browser was closed when the run ended. This cell can be rerun later to download them. If a Drive error interrupted final copying, a completed local ZIP can still be recovered *while that Colab runtime survives*. After a hard reset, regenerate the exports from the restored snapshot.

In [ ]:
# @title Current progress and next retry
from datetime import datetime, timezone

STATUS = client.json("status")
s = STATUS["stats"]
print(f"Domains: {s['total']:,} | Processed: {s['terminal']:,} | Websites with email: {s['with_email']:,}")
print(f"Unique addresses: {s['unique_emails']:,} | Website-email pairs: {s['site_email_pairs']:,} | Pages: {s['pages_scanned']:,}")
print("Outcomes:", json.dumps(s["statuses"], indent=2))
if s.get("next_retry_at"):
    print("Next retry eligible:", datetime.fromtimestamp(s["next_retry_at"], timezone.utc).isoformat())
if STATUS.get("last_checkpoint"):
    last = STATUS["last_checkpoint"]
    print("Last verified snapshot:", datetime.fromtimestamp(last["created_at"], timezone.utc).isoformat())
    print("Snapshot manifest:", last["manifest"])
print("Mode:", s["mode"], "| Worker:", s["worker_state"])
if s.get("worker_error"):
    print("WORKER NEEDS ATTENTION:", s["worker_error"])

In [ ]:
# @title Export / download a ZIP
REFRESH_EXPORTS = False  # @param {type:"boolean"}
DOWNLOAD_ZIP = True  # @param {type:"boolean"}
SHOW_FIRST_TEN_EMAILS = False  # @param {type:"boolean"}

marker = LOCAL_PROJECT / "last-export.json"
if REFRESH_EXPORTS or not marker.is_file():
    RESULTS = client.json("export")
else:
    RESULTS = json.loads(marker.read_text())
archive = Path(RESULTS["export_zip"])
if not archive.is_file():
    RESULTS = client.json("export")
    archive = Path(RESULTS["export_zip"])
print("Local result ZIP:", archive)
print("Size:", round(archive.stat().st_size / 1024**2, 2), "MiB")
print("Private Drive copy:", RESULTS.get("drive_export_zip") or "Not copied to Drive; keep this runtime alive and download now.")
if SHOW_FIRST_TEN_EMAILS:
    import csv
    import itertools
    with (Path(RESULTS["export_dir"]) / "emails.csv").open(encoding="utf-8-sig", newline="") as source:
        for row in itertools.islice(csv.DictReader(source), 10):
            print(row["domain"], row["email"], row["source_url"])
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive))

## Optional: a deliberate deeper pass

Do not requeue just because you have not reached an arbitrary email count. Review `coverage.csv`, then increase limits where the report justifies it. A continuation uses the saved frontier; a **fresh refetch** also re-requests the homepage and retains historical evidence. Raising a sitemap/document/frontier cap may require a fresh refetch to rediscover skipped/discarded candidates.

This cell is **off by default** so Runtime → Run all cannot silently requeue outcomes. If you enable it, review its claim limit and new page/depth budgets, run it once, then select pilot/full in the Start / resume cell again. Requeueing blocked sites does not turn off robots, cooldowns, or access checks.

In [ ]:
# @title Optional: requeue a reviewed set of gaps
REQUEUE_GAPS = False  # @param {type:"boolean"}
REQUEUE_STATUS = "no_email"  # @param ["no_email", "failed", "blocked"]
REQUEUE_LIMIT = 100  # @param {type:"integer"}
FRESH_REFETCH = False  # @param {type:"boolean"}
SECOND_PASS_PAGES = 60  # @param {type:"integer"}
SECOND_PASS_DEPTH = 4  # @param {type:"integer"}

if REQUEUE_GAPS:
    args = ["retry", "--status", REQUEUE_STATUS, "--limit", str(REQUEUE_LIMIT)]
    if FRESH_REFETCH:
        args.append("--fresh")
    print(json.dumps(client.json(*args), indent=2))
    MAX_PAGES_PER_SITE = SECOND_PASS_PAGES
    MAX_DEPTH = SECOND_PASS_DEPTH
    print("Requeued only; no crawl started. Now rerun Start / resume to use the deeper budgets.")
else:
    print("No automatic requeue. Review coverage before enabling a second pass.")

## Recovery and troubleshooting

- **Colab reset / lost connection:** use the same project name, mount Drive again, run setup/restore/import. Select full in Start / resume. Already saved evidence is not erased.
- **Another runtime/session marker:** stop the other runtime. If it died, wait up to 15 minutes for the advisory marker to expire. Never delete a marker just to run concurrently.
- **Drive quota / mount / I/O failure:** the runner stops rather than silently claiming continuing protection. Download the local ZIP if possible, fix Drive, and rerun. Keep previous snapshot `.db` + `.json` pairs together. Do not copy an active database/WAL to Drive manually.
- **Corrupt latest snapshot:** recovery tries older verified generations. If all are invalid, it refuses an empty reset. Keep those files; do not delete your project to clear the message.
- **Low space:** delete only older result ZIPs/exports you have safely copied elsewhere. Three complete DB generations plus staging require several times the DB's size. A Drive volume/quota is not an off-account backup.
- **Browser installation/launch error:** rerun installation and the environment check, or disable optional browser fallback and rerun those cells. Missing Chromium must not turn the entire input into failed sites.
- **No public email / access block:** inspect coverage/evidence; do not assume a bug or attempt a bypass. Some sites cannot provide an email to this scraper.
- **Privacy:** the notebook/Drive project can contain business metadata and personal addresses. Keep it private, protect your account, clear outputs before sharing, and set suitable retention/marketing-consent policies.

For detailed limitations, see the repository's `docs/COLAB.md`, `README.md`, and `SECURITY.md`. A continuously supervised VPS remains an alternative if you later need stronger always-on guarantees, but is **not required to use this notebook**.